# 🚗 Chicago Car Crash Interpretability Analysis

## 🧠 Predicting Causes of Traffic Crashes Using Machine Learning

---

# 📊 Business Understanding

---

## 🌍 1. Introduction: Real-World Problem

Traffic accidents are a major public safety issue in urban areas such as Chicago.

Each crash is influenced by multiple factors including:
- Road conditions  
- Weather conditions  
- Vehicle types  
- Human behavior  

Understanding these factors is essential for reducing accidents and improving road safety outcomes.

---

## 🎯 2. Problem Statement

The goal of this project is to develop a machine learning model that predicts the **primary contributory cause of a traffic crash** using information from crash, vehicle, and people datasets.

This is a **multi-class classification problem**, where the model learns patterns that link crash conditions to their likely causes.

---

## 👥 3. Stakeholders and Use Cases

This project is useful for several real-world stakeholders:

### 🏛️ City of Chicago Department of Transportation
- Identify high-risk locations and conditions  
- Improve road infrastructure planning  

### 🚦 Traffic Safety Authorities
- Design targeted road safety campaigns  
- Reduce accident frequency  

### 🛡️ Insurance Companies
- Improve risk assessment models  
- Optimize pricing and claims evaluation  

### 🏗️ Urban Planners
- Improve road design and traffic flow  
- Reduce congestion-related risks  

### 🚑 Emergency Response Teams
- Improve resource allocation  
- Optimize emergency response times  

---

## ❓ 4. Why This Matters

This analysis helps answer key questions such as:

- What conditions most often lead to traffic accidents?  
- Can crash causes be predicted using available data?  
- Which factors contribute most to accident severity?  

Understanding these patterns can support **data-driven decision-making** in public safety.

---

## 📌 5. Project Value (Conclusion)

Although this is a **proof of concept**, it demonstrates how machine learning can be used to uncover patterns in traffic crash data.

The insights generated can help stakeholders:
- Reduce accident rates  
- Improve road safety strategies  
- Make informed infrastructure decisions  
- Support public safety planning  

Ultimately, this project shows how data science can contribute to **safer cities and better transportation systems**.

# 📊 Data Understanding

This section explores the structure, source, and properties of the datasets used in this project. The goal is to evaluate whether the data is suitable for predicting the primary contributory cause of traffic crashes in Chicago and to understand its real-world relevance.

## 📁 Data Sources and Subset Strategy

This project uses data from the City of Chicago Traffic Crash Data Portal, which includes three related datasets:

- Traffic Crashes dataset (event-level crash information)
- Vehicles dataset (vehicle involvement data)
- People dataset (driver and occupant information)

All datasets are linked using a common key: `CRASH_RECORD_ID`.

### 📦 Subset Strategy

Instead of using the full datasets (which contain over 1 million records in total), this project uses a **reduced subset of approximately 30MB per dataset**.

This subset approach is used to:
- Improve computational efficiency during analysis
- Enable faster iteration during exploration and feature selection
- Maintain a representative sample of real-world crash patterns

This ensures the project remains practical while preserving meaningful structure in the data.

In [3]:

import os
import urllib.request
import ssl

print("📥 Initializing secure download pipeline bypass...")

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs('images', exist_ok=True)

# Fixed Socrata API endpoints ordered by CRASH_RECORD_ID to guarantee overlapping keys!
URLS = {
    'crashes.csv':  'https://data.cityofchicago.org/resource/85ca-t3if.csv?$limit=100000&$order=CRASH_RECORD_ID',
    'vehicles.csv': 'https://data.cityofchicago.org/resource/68nd-jvt3.csv?$limit=200000&$order=CRASH_RECORD_ID',
    'people.csv':   'https://data.cityofchicago.org/resource/u6pd-qa9d.csv?$limit=200000&$order=CRASH_RECORD_ID',
}

# 🛠️ THE FIX: Create an unverified SSL context to bypass local issuer certificate check
ssl_context = ssl._create_unverified_context()

for filename, url in URLS.items():
    filepath = os.path.join(DATA_DIR, filename)
    if os.path.exists(filepath):
        print(f'[SKIP] {filename} already exists ({os.path.getsize(filepath)/1e6:.1f} MB)')
    else:
        print(f'[DOWNLOADING] {filename} ...')
        
        # Pass the unverified context directly to urlretrieve
        opener = urllib.request.build_opener(urllib.request.HTTPSHandler(context=ssl_context))
        urllib.request.install_opener(opener)
        
        urllib.request.urlretrieve(url, filepath)
        size_mb = os.path.getsize(filepath) / 1e6
        print(f'[DONE] {filename} saved ({size_mb:.1f} MB)')

print("\n🚀 All aligned datasets downloaded successfully without SSL blocks!")

📥 Initializing secure download pipeline bypass...
[DOWNLOADING] crashes.csv ...
[DONE] crashes.csv saved (63.1 MB)
[DOWNLOADING] vehicles.csv ...
[DONE] vehicles.csv saved (71.4 MB)
[DOWNLOADING] people.csv ...
[DONE] people.csv saved (68.5 MB)

🚀 All aligned datasets downloaded successfully without SSL blocks!


In [6]:
import pandas as pd

crashes = pd.read_csv("data/crashes.csv")
vehicles = pd.read_csv("data/vehicles.csv")
people = pd.read_csv("data/people.csv")

print("Crashes:", crashes.shape)
print("Vehicles:", vehicles.shape)
print("People:", people.shape)

c:\Users\admin\anaconda3\envs\learn-env\lib\site-packages\IPython\core\interactiveshell.py:3145: DtypeWarning: Columns (18,39,40,41,43,47,48,49,52,54,57,58,60) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


Crashes: (100000, 49)
Vehicles: (200000, 71)
People: (200000, 29)


In [7]:
crash_ids = set(crashes["crash_record_id"].astype(str))

vehicle_overlap = len(
    crash_ids.intersection(
        set(vehicles["crash_record_id"].astype(str))
    )
)

people_overlap = len(
    crash_ids.intersection(
        set(people["crash_record_id"].astype(str))
    )
)

print("Crash-Vehicle overlap:", vehicle_overlap)
print("Crash-People overlap:", people_overlap)

Crash-Vehicle overlap: 98040
Crash-People overlap: 91069


In [9]:
# Preview the Traffic Crashes dataset

print("Traffic Crashes Dataset")
print("Shape:", crashes.shape)

crashes.head()

Traffic Crashes Dataset
Shape: (100000, 49)


,crash_record_id,crash_date_est_i,crash_date,posted_speed_limit,traffic_control_device,device_condition,weather_condition,lighting_condition,first_crash_type,trafficway_type,...,injuries_non_incapacitating,injuries_reported_not_evident,injuries_no_indication,injuries_unknown,crash_hour,crash_day_of_week,idot_control_no,latitude,longitude,location
0,000013b0123279411e0ec856dae95ab9f0851764350b7f...,NaN,2020-11-16T13:50:00.000,35,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,PARKED MOTOR VEHICLE,PARKING LOT,...,0.0,0.0,1.0,0.0,13,2,X002073980,41.844584,-87.695360,POINT (-87.695359843215 41.844584300311)
1,00002c0771fb6f2c70ba775b7f6b501608cadea85c1dd1...,NaN,2016-04-16T05:49:00.000,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,DAWN,SIDESWIPE SAME DIRECTION,OTHER,...,0.0,0.0,2.0,0.0,5,7,X000473010,41.844734,-87.695363,POINT (-87.695363066709 41.844733938666)
2,000043c6564ec4d54bc4efd957d97ca97f38a965dd64b4...,Y,2019-12-22T02:11:00.000,30,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",FIXED OBJECT,NOT DIVIDED,...,0.0,0.0,1.0,0.0,2,1,X001788722,41.691319,-87.540255,POINT (-87.540255297708 41.691318518437)
3,00005696946846c8b8a1d378dba4e2a5ed84a9b2876fe0...,NaN,2024-02-02T09:48:00.000,30,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,FIXED OBJECT,OTHER,...,0.0,0.0,1.0,0.0,9,6,X003310805,41.811261,-87.606750,POINT (-87.606749972034 41.811261436387)
4,000070ed7a6357c3298f5edc6fb7d5ce925a10f46660f3...,NaN,2016-10-24T02:42:00.000,35,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",SIDESWIPE SAME DIRECTION,DIVIDED - W/MEDIAN (NOT RAISED),...,0.0,0.0,2.0,0.0,2,2,X000625684,41.873272,-87.695937,POINT (-87.69593723519 41.873272095065)


In [10]:
# Preview the Crash Vehicles dataset

print("Crash Vehicles Dataset")
print("Shape:", vehicles.shape)

vehicles.head()

Crash Vehicles Dataset
Shape: (200000, 71)


,crash_unit_id,crash_record_id,crash_date,unit_no,unit_type,num_passengers,vehicle_id,cmrc_veh_i,make,model,...,trailer1_length,trailer2_length,total_vehicle_length,axle_cnt,vehicle_config,cargo_body_type,load_type,hazmat_out_of_service_i,mcs_out_of_service_i,hazmat_class
0,995163,000013b0123279411e0ec856dae95ab9f0851764350b7f...,2020-11-16T13:50:00.000,1,DRIVER,NaN,943049.0,NaN,HYUNDAI,SONATA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,995164,000013b0123279411e0ec856dae95ab9f0851764350b7f...,2020-11-16T13:50:00.000,2,PARKED,NaN,943051.0,NaN,CHEVROLET,TRAVERSE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,40893,00002c0771fb6f2c70ba775b7f6b501608cadea85c1dd1...,2016-04-16T05:49:00.000,1,DRIVER,NaN,39524.0,NaN,LINCOLN-CONTINENTAL,MKZ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,40894,00002c0771fb6f2c70ba775b7f6b501608cadea85c1dd1...,2016-04-16T05:49:00.000,2,DRIVER,NaN,39526.0,NaN,HYUNDAI,SONATA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,807770,000043c6564ec4d54bc4efd957d97ca97f38a965dd64b4...,2019-12-22T02:11:00.000,1,DRIVER,NaN,767006.0,NaN,LINCOLN-CONTINENTAL,TOWNCAR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# Preview the Crash People dataset

print("Crash People Dataset")
print("Shape:", people.shape)

people.head()

Crash People Dataset
Shape: (200000, 29)


,person_id,person_type,crash_record_id,vehicle_id,crash_date,seat_no,city,state,zipcode,sex,...,ems_run_no,driver_action,driver_vision,physical_condition,pedpedal_action,pedpedal_visibility,pedpedal_location,bac_result,bac_result_value,cell_phone_use
0,O995163,DRIVER,000013b0123279411e0ec856dae95ab9f0851764350b7f...,943049.0,2020-11-16T13:50:00.000,NaN,CHICAGO,IL,60642,F,...,NaN,IMPROPER PARKING,UNKNOWN,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
1,O40893,DRIVER,00002c0771fb6f2c70ba775b7f6b501608cadea85c1dd1...,39524.0,2016-04-16T05:49:00.000,NaN,NaN,NaN,NaN,X,...,NaN,IMPROPER LANE CHANGE,UNKNOWN,UNKNOWN,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
2,O40894,DRIVER,00002c0771fb6f2c70ba775b7f6b501608cadea85c1dd1...,39526.0,2016-04-16T05:49:00.000,NaN,CHICAGO,IL,60639,F,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
3,O807770,DRIVER,000043c6564ec4d54bc4efd957d97ca97f38a965dd64b4...,767006.0,2019-12-22T02:11:00.000,NaN,NaN,NaN,NaN,X,...,NaN,UNKNOWN,UNKNOWN,UNKNOWN,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
4,O1752154,DRIVER,00005696946846c8b8a1d378dba4e2a5ed84a9b2876fe0...,1668221.0,2024-02-02T09:48:00.000,NaN,CHICAGO,IL,60633,M,...,NaN,EMERGENCY VEHICLE ON CALL,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN


In [12]:
sample_id = crashes["crash_record_id"].iloc[0]

print("Sample crash ID:", sample_id)

print("\nIn vehicles:")
print(sample_id in set(vehicles["crash_record_id"]))

print("\nIn people:")
print(sample_id in set(people["crash_record_id"]))

Sample crash ID: 000013b0123279411e0ec856dae95ab9f0851764350b7feaeb982c7707c6722066910e9391e60f45cec4b7a7a6643eeedb5de39e7245b03447a44c793680dc4b

In vehicles:
True

In people:
True


In [8]:
# 3 checking data shapes
# Check dataset sizes (important for understanding scale)

print("Crashes shape:", crashes.shape)
print("Vehicles shape:", vehicles.shape)
print("People shape:", people.shape)

Crashes shape: (100000, 49)
Vehicles shape: (200000, 71)
People shape: (200000, 29)


In [110]:
# 4 preview datasets
#4.1 crash dataset

crashes.head()

,crash_record_id,crash_date_est_i,crash_date,posted_speed_limit,traffic_control_device,device_condition,weather_condition,lighting_condition,first_crash_type,trafficway_type,...,injuries_non_incapacitating,injuries_reported_not_evident,injuries_no_indication,injuries_unknown,crash_hour,crash_day_of_week,idot_control_no,latitude,longitude,location
0,419e0f4348b4274d902d84bd7aa58f5fa9c10820b77abc...,NaN,2026-05-19T23:15:00.000,45,STOP SIGN/FLASHER,FUNCTIONING IMPROPERLY,CLEAR,DARKNESS,TURNING,DIVIDED - W/MEDIAN (NOT RAISED),...,0.0,0.0,2.0,0.0,23,3,X004237573,41.781851,-87.575475,POINT (-87.575474638028 41.781851295939)
1,1471094d2f2a497653584d887ed73cddcdfaa35fd02b20...,NaN,2026-05-20T07:51:00.000,30,STOP SIGN/FLASHER,FUNCTIONING PROPERLY,CLEAR,DAYLIGHT,PEDESTRIAN,FOUR WAY,...,2.0,0.0,2.0,0.0,7,4,X004237669,41.894349,-87.662208,POINT (-87.662207823341 41.89434904862)
2,1c99690e19bd557cc6ad0674668dc492ab979c4e63a279...,Y,2026-05-20T02:00:00.000,30,UNKNOWN,UNKNOWN,CLEAR,"DARKNESS, LIGHTED ROAD",FIXED OBJECT,NOT DIVIDED,...,0.0,0.0,1.0,0.0,2,4,X004237255,41.759085,-87.567448,POINT (-87.567447849458 41.759085475865)
3,401b13002f631a3faa4af328d3896d70547210e3cfc218...,NaN,2026-05-20T07:50:00.000,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,DAYLIGHT,SIDESWIPE SAME DIRECTION,FOUR WAY,...,0.0,0.0,3.0,0.0,7,4,X004237570,41.793371,-87.684175,POINT (-87.684175376759 41.79337055433)
4,d4617f749ee48ca7b2b74cd96171a7ab3d786cf2fcbda5...,NaN,2026-05-18T07:45:00.000,30,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,PARKED MOTOR VEHICLE,NOT DIVIDED,...,0.0,0.0,1.0,0.0,7,2,X004237264,41.654285,-87.603500,POINT (-87.603500025118 41.654285003672)


In [111]:
# 4.2 vehicles dataset

vehicles.head()

,crash_unit_id,crash_record_id,crash_date,unit_no,unit_type,num_passengers,vehicle_id,cmrc_veh_i,make,model,...,trailer1_length,trailer2_length,total_vehicle_length,axle_cnt,vehicle_config,cargo_body_type,load_type,hazmat_out_of_service_i,mcs_out_of_service_i,hazmat_class
0,2322519,435043b3192b9c0b69eb1c6cc0f856e74d5aab769746c1...,2026-06-09T02:34:00.000,1,DRIVER,NaN,2215120.0,NaN,CHEVROLET,IMPALA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2322520,435043b3192b9c0b69eb1c6cc0f856e74d5aab769746c1...,2026-06-09T02:34:00.000,2,DRIVER,1.0,2215122.0,NaN,MAZDA,CX-5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2322507,fe1c7a79eab83a76e37fb858d57715d0fff5a96f11663b...,2026-06-09T00:31:00.000,1,DRIVER,3.0,2215107.0,NaN,CHEVROLET,EQUINOX,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2322508,fe1c7a79eab83a76e37fb858d57715d0fff5a96f11663b...,2026-06-09T00:31:00.000,2,DRIVER,NaN,2215109.0,NaN,BUICK,ENCORE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2322506,ce927aac11ea2e98f2346c1c32c2dfa92b8850b8639648...,2026-06-09T00:26:00.000,1,DRIVER,NaN,2215106.0,NaN,CADILLAC,XT5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [112]:
# 4.3 people dataset

people.head()

,person_id,person_type,crash_record_id,vehicle_id,crash_date,seat_no,city,state,zipcode,sex,...,ems_run_no,driver_action,driver_vision,physical_condition,pedpedal_action,pedpedal_visibility,pedpedal_location,bac_result,bac_result_value,cell_phone_use
0,O2322519,DRIVER,435043b3192b9c0b69eb1c6cc0f856e74d5aab769746c1...,2215120.0,2026-06-09T02:34:00.000,NaN,CHICAGI,IL,60628.0,M,...,NaN,FAILED TO YIELD,UNKNOWN,UNKNOWN,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
1,O2322520,DRIVER,435043b3192b9c0b69eb1c6cc0f856e74d5aab769746c1...,2215122.0,2026-06-09T02:34:00.000,NaN,HAMMOND,IN,46327.0,F,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
2,P511045,PASSENGER,435043b3192b9c0b69eb1c6cc0f856e74d5aab769746c1...,2215122.0,2026-06-09T02:34:00.000,3.0,HAMMOND,IN,46327.0,F,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,O2322507,DRIVER,fe1c7a79eab83a76e37fb858d57715d0fff5a96f11663b...,2215107.0,2026-06-09T00:31:00.000,NaN,CHICAGO,IL,60634.0,F,...,NaN,IMPROPER LANE CHANGE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
4,O2322508,DRIVER,fe1c7a79eab83a76e37fb858d57715d0fff5a96f11663b...,2215109.0,2026-06-09T00:31:00.000,NaN,CHICAGO,IL,60612.0,M,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN


In [113]:
# checking column names for better understanding of data structure
print("Crashes columns:", crashes.columns)


Crashes columns: Index(['crash_record_id', 'crash_date_est_i', 'crash_date',
       'posted_speed_limit', 'traffic_control_device', 'device_condition',
       'weather_condition', 'lighting_condition', 'first_crash_type',
       'trafficway_type', 'lane_cnt', 'alignment', 'roadway_surface_cond',
       'road_defect', 'report_type', 'crash_type', 'intersection_related_i',
       'private_property_i', 'hit_and_run_i', 'damage', 'date_police_notified',
       'prim_contributory_cause', 'sec_contributory_cause', 'street_no',
       'street_direction', 'street_name', 'beat_of_occurrence',
       'photos_taken_i', 'statements_taken_i', 'dooring_i', 'work_zone_i',
       'work_zone_type', 'workers_present_i', 'num_units', 'crash_month',
       'most_severe_injury', 'injuries_total', 'injuries_fatal',
       'injuries_incapacitating', 'injuries_non_incapacitating',
       'injuries_reported_not_evident', 'injuries_no_indication',
       'injuries_unknown', 'crash_hour', 'crash_day_of_week',
  

In [114]:
# 6 cleaning column names for easier access
crashes.columns = crashes.columns.str.strip()


print("Cleaned Crashes columns:", crashes.columns)


Cleaned Crashes columns: Index(['crash_record_id', 'crash_date_est_i', 'crash_date',
       'posted_speed_limit', 'traffic_control_device', 'device_condition',
       'weather_condition', 'lighting_condition', 'first_crash_type',
       'trafficway_type', 'lane_cnt', 'alignment', 'roadway_surface_cond',
       'road_defect', 'report_type', 'crash_type', 'intersection_related_i',
       'private_property_i', 'hit_and_run_i', 'damage', 'date_police_notified',
       'prim_contributory_cause', 'sec_contributory_cause', 'street_no',
       'street_direction', 'street_name', 'beat_of_occurrence',
       'photos_taken_i', 'statements_taken_i', 'dooring_i', 'work_zone_i',
       'work_zone_type', 'workers_present_i', 'num_units', 'crash_month',
       'most_severe_injury', 'injuries_total', 'injuries_fatal',
       'injuries_incapacitating', 'injuries_non_incapacitating',
       'injuries_reported_not_evident', 'injuries_no_indication',
       'injuries_unknown', 'crash_hour', 'crash_day_of_w

In [115]:
# 7 data structure and types
# Understand data types and missing values

crashes.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 49 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   crash_record_id                1000 non-null   object 
 1   crash_date_est_i               61 non-null     object 
 2   crash_date                     1000 non-null   object 
 3   posted_speed_limit             1000 non-null   int64  
 4   traffic_control_device         1000 non-null   object 
 5   device_condition               1000 non-null   object 
 6   weather_condition              1000 non-null   object 
 7   lighting_condition             1000 non-null   object 
 8   first_crash_type               1000 non-null   object 
 9   trafficway_type                1000 non-null   object 
 10  lane_cnt                       21 non-null     float64
 11  alignment                      1000 non-null   object 
 12  roadway_surface_cond           1000 non-null   ob

In [116]:
# 8 missing values in crashes dataset
crashes.isna().sum().sort_values(ascending=False).head(10)

workers_present_i     999
dooring_i             999
work_zone_type        995
photos_taken_i        992
work_zone_i           992
lane_cnt              979
statements_taken_i    973
private_property_i    940
crash_date_est_i      939
longitude             865
dtype: int64

In [117]:
# 9 statistical summary of numeric + categorical features
crashes.describe(include="all").T.head(15)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
crash_record_id,1000,1000,ad68a5497001f7a5c081cf76941314adcd14de4f1b6704...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
crash_date_est_i,61,2,Y,51,NaN,NaN,NaN,NaN,NaN,NaN,NaN
crash_date,1000,994,2021-04-05T21:29:00.000,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
posted_speed_limit,1000,NaN,NaN,NaN,26.498,9.77167,0,20,30,30,60
traffic_control_device,1000,12,NO CONTROLS,626,NaN,NaN,NaN,NaN,NaN,NaN,NaN
device_condition,1000,6,NO CONTROLS,634,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weather_condition,1000,9,CLEAR,780,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lighting_condition,1000,6,DAYLIGHT,612,NaN,NaN,NaN,NaN,NaN,NaN,NaN
first_crash_type,1000,17,SIDESWIPE SAME DIRECTION,215,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trafficway_type,1000,18,ONE-WAY,222,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [118]:
# 10 Check top crash causes (correct column name)

crashes["prim_contributory_cause"].value_counts().head(10)

UNABLE TO DETERMINE                    350
FOLLOWING TOO CLOSELY                   94
FAILING TO YIELD RIGHT-OF-WAY           91
NOT APPLICABLE                          73
IMPROPER BACKING                        58
IMPROPER LANE USAGE                     54
DRIVING SKILLS/KNOWLEDGE/EXPERIENCE     44
IMPROPER OVERTAKING/PASSING             42
WEATHER                                 34
IMPROPER TURNING/NO SIGNAL              34
Name: prim_contributory_cause, dtype: int64

In [119]:
target_col = "prim_contributory_cause"

if target_col in crashes.columns:
    print(crashes[target_col].value_counts().head(10))
else:
    print("Target column not found.")

UNABLE TO DETERMINE                    350
FOLLOWING TOO CLOSELY                   94
FAILING TO YIELD RIGHT-OF-WAY           91
NOT APPLICABLE                          73
IMPROPER BACKING                        58
IMPROPER LANE USAGE                     54
DRIVING SKILLS/KNOWLEDGE/EXPERIENCE     44
IMPROPER OVERTAKING/PASSING             42
WEATHER                                 34
IMPROPER TURNING/NO SIGNAL              34
Name: prim_contributory_cause, dtype: int64


## 🎯 Target Variable

The target variable is `prim_contributory_cause`, which represents the primary reason assigned to each crash.

This is a multi-class classification problem where we aim to predict crash causes based on environmental and road conditions.

## 🧹 3. Data Preparation & Feature Engineering

In this stage, we combine the three datasets (crashes, vehicles, and people) into a single analytical dataset.

We then clean missing values, handle inconsistencies, and prepare a machine learning-ready dataset for predicting crash causes.

In [120]:
import numpy as np
import pandas as pd

# 1. Clean column names across all datasets
crashes.columns = crashes.columns.str.strip().str.lower()
vehicles.columns = vehicles.columns.str.strip().str.lower()
people.columns = people.columns.str.strip().str.lower()

print("Columns cleaned.")

Columns cleaned.


In [121]:
for df in [crashes, vehicles, people]:
    df["crash_record_id"] = df["crash_record_id"].astype(str).str.strip().str.upper()

In [122]:
# standardize keys in all datasets for merging
crashes["crash_record_id"] = crashes["crash_record_id"].astype(str).str.strip()
vehicles["crash_record_id"] = vehicles["crash_record_id"].astype(str).str.strip()
people["crash_record_id"] = people["crash_record_id"].astype(str).str.strip()

In [123]:
print("Crashes IDs:", crashes["crash_record_id"].head())
print("People IDs:", people["crash_record_id"].head())
print("Vehicles IDs:", vehicles["crash_record_id"].head())

Crashes IDs: 0    419E0F4348B4274D902D84BD7AA58F5FA9C10820B77ABC...
1    1471094D2F2A497653584D887ED73CDDCDFAA35FD02B20...
2    1C99690E19BD557CC6AD0674668DC492AB979C4E63A279...
3    401B13002F631A3FAA4AF328D3896D70547210E3CFC218...
4    D4617F749EE48CA7B2B74CD96171A7AB3D786CF2FCBDA5...
Name: crash_record_id, dtype: object
People IDs: 0    435043B3192B9C0B69EB1C6CC0F856E74D5AAB769746C1...
1    435043B3192B9C0B69EB1C6CC0F856E74D5AAB769746C1...
2    435043B3192B9C0B69EB1C6CC0F856E74D5AAB769746C1...
3    FE1C7A79EAB83A76E37FB858D57715D0FFF5A96F11663B...
4    FE1C7A79EAB83A76E37FB858D57715D0FFF5A96F11663B...
Name: crash_record_id, dtype: object
Vehicles IDs: 0    435043B3192B9C0B69EB1C6CC0F856E74D5AAB769746C1...
1    435043B3192B9C0B69EB1C6CC0F856E74D5AAB769746C1...
2    FE1C7A79EAB83A76E37FB858D57715D0FFF5A96F11663B...
3    FE1C7A79EAB83A76E37FB858D57715D0FFF5A96F11663B...
4    CE927AAC11EA2E98F2346C1C32C2DFA92B8850B8639648...
Name: crash_record_id, dtype: object


In [124]:
crash_ids = set(crashes["crash_record_id"])

print("Overlap crashes-vehicles:", len(crash_ids & set(vehicles["crash_record_id"])))
print("Overlap crashes-people:", len(crash_ids & set(people["crash_record_id"])))

Overlap crashes-vehicles: 0
Overlap crashes-people: 0


In [125]:
common_ids = set(crashes["crash_record_id"])

vehicles = vehicles[vehicles["crash_record_id"].isin(common_ids)]
people = people[people["crash_record_id"].isin(common_ids)]

In [126]:
# 2. Count number of vehicles per crash

veh_agg = vehicles.groupby("crash_record_id").agg(
    total_vehicles=("vehicle_id", "count")
).reset_index()

veh_agg.head()

,crash_record_id,total_vehicles


In [127]:
# 3. Aggregate people data

people_agg = people.groupby("crash_record_id").agg(
    avg_driver_age=("age", "mean"),
).reset_index()

people_agg.head()

,crash_record_id,avg_driver_age


In [128]:
# 4. Merging datasets to create a master dataset for modeling

df = crashes.copy()

df = df.merge(veh_agg, on="crash_record_id", how="left")
df = df.merge(people_agg, on="crash_record_id", how="left")

print(df[["total_vehicles", "avg_driver_age"]].isna().sum())

total_vehicles    1000
avg_driver_age    1000
dtype: int64


In [129]:
print(crashes["crash_record_id"].nunique())
print(vehicles["crash_record_id"].nunique())
print(people["crash_record_id"].nunique())

1000
0
0


In [130]:
# 5 handling missing values in the merged dataset
# Missing vehicle count = assume one vehicle
df["total_vehicles"] = df["total_vehicles"].fillna(1)

# Missing age = median age
df["avg_driver_age"] = df["avg_driver_age"].fillna(
    df["avg_driver_age"].median()
)

df[["total_vehicles", "avg_driver_age"]].isna().sum()

c:\Users\admin\anaconda3\envs\learn-env\lib\site-packages\numpy\lib\nanfunctions.py:1113: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


total_vehicles       0
avg_driver_age    1000
dtype: int64

In [131]:
# 6 checking if marge worked correctly by comparing some crash record ids
df[["total_vehicles", "avg_driver_age"]].describe()
print(df.shape)

(1000, 51)
